# CaliDoc: WACV 2027 Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHHamdan/CaliDoc/blob/main/notebooks/CaliDoc_WACV2027_Demo.ipynb) ·
[GitHub](https://github.com/MHHamdan/CaliDoc) ·
[Models](https://huggingface.co/collections/MHamdan/calidoc-wacv-2027-6ad41a62c6a81846dfeb44b4) ·
[Supplement](https://github.com/MHHamdan/CaliDoc/releases/download/wacv2027-camera-ready/CaliDoc_WACV2027_Supplementary.pdf)

## What CaliDoc does

Document AI systems deliver **complete fields** (a total, a date, a merchant name), but
uncertainty is usually calibrated per token. A model can look well calibrated token by
token while missing whole fields.

CaliDoc calibrates the complete field instead. It wraps a key-information-extraction model
with a conformal risk-control step that bounds the expected fraction of missed fields per
document. Each extracted field is then routed: **pass** it on, send it to **review**, or
**abstain**.

This notebook runs the released WACV 2027 checkpoints on real benchmark receipts and on a
document you upload. It needs only a standard Colab runtime, with no account, token or API
key.

<small>CaliDoc's guarantee assumes exchangeability with the calibration distribution, and the
released checkpoints are benchmark-specific. Automatic OCR in this public demo is provided
for convenience and differs from the benchmark preprocessing used in the WACV
experiments.</small>

## Step 1: Install

This step clones the public repository and installs CaliDoc with its demo extras (Gradio
and docTR for local OCR). It then loads the demo logic from `demo/gradio/app.py`, the
same code the interactive app runs. The released CORD model (1.4 GB) downloads
anonymously from Hugging Face. A GPU runtime is optional.

In [ ]:
!git clone -q --depth 1 https://github.com/MHHamdan/CaliDoc.git CaliDoc
%pip install -q "./CaliDoc[demo]"

import sys
sys.path.insert(0, "CaliDoc/demo/gradio")
import app  # demo/gradio/app.py: example loading, OCR, inference call, presentation

import pandas as pd
import torch
from IPython.display import HTML, Markdown, display

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
cord = app.predictor(app.MODELS["CORD (receipts)"])   # MHamdan/calidoc-layoutlmv3-large-cord
print("alpha =", cord.config["alpha"], "| pooled threshold =", round(cord.pooled_lambda, 4))


def show(vis, summary, rows, what, gt=""):
    display(HTML(summary))
    display(vis)
    display(HTML(app.LEGEND_HTML))
    display(pd.DataFrame(rows, columns=app.TABLE_HEADERS).set_index("#"))
    display(Markdown(what))
    if gt:
        display(Markdown(gt))

## Step 2: Try a real WACV example

`cord_test_0042` is a CORD v2 test receipt (CC BY 4.0, Park et al. 2019), downloaded from
[`MHamdan/calidoc-demo-examples`](https://huggingface.co/datasets/MHamdan/calidoc-demo-examples).
It was chosen by a fixed rule (median word count), not by prediction quality, and uses the
benchmark's own OCR.

In [ ]:
vis, summary, rows, what, gt, result_0042 = app.run_example("CORD: Representative receipt")
show(vis, summary, rows, what, gt)

All 9 extracted fields are correct, yet all 9 are routed for review. On CORD, the supported
field type `MENU.SUB.CNT` has calibrated threshold λ = 1.0, so its label enters every
word's prediction set. CORD is CaliDoc's most expensive setting, and the WACV paper reports
it as such.

## Step 3: Why review routing matters

`cord_test_0014` is a denser receipt (90th-percentile word count). One of its 25
extractions is wrong: *Coupon 100,000* is predicted `SUB_TOTAL.ETC`, but the CORD gold label
is `TOTAL.TOTAL_ETC`. CaliDoc does not present it as a confident final field. Its calibrated
set excludes the prediction, so it is the only field in state **conflict**, and it is routed
for review. The ground-truth check is shown separately: routing itself uses no labels.

In [ ]:
vis, summary, rows, what, gt, result_0014 = app.run_example("CORD: Review-routing case")
show(vis, summary, rows, what, gt)
[{k: f[k] for k in ("type", "text", "set_sizes", "threshold", "threshold_source", "status")}
 for f in result_0014["fields"] if f["status"] != "ambiguous_review"]

## Step 4: Upload your own document

Upload a PNG, JPEG or WEBP image of a receipt or form. The pipeline is: image, then docTR
OCR (locally, in this runtime), then the released Hugging Face checkpoint, then CaliDoc
calibration, then visualization. No OCR file is needed.

This is an **exploratory user-upload result**. Uploaded documents do not receive the
paper's formal guarantee, and *pass* does not certify that a field is correct. Your image
stays in this Colab runtime.

If you cancel the upload, or run the notebook outside Colab, the cell uses the
review-routing receipt instead, re-encoded as a JPEG photo.

In [ ]:
import os, tempfile

MODEL = "CORD (receipts)"   # or "WildReceipt (receipts)", "FUNSD (forms)", "SROIE (receipts, case study)"
my_image = None
try:
    from google.colab import files
    uploaded = files.upload()   # opens a file picker
    if uploaded:
        name, data = next(iter(uploaded.items()))
        my_image = os.path.join(tempfile.mkdtemp(), os.path.basename(name))
        with open(my_image, "wb") as fh:
            fh.write(data)
except ImportError:
    pass
if my_image is None:
    sample, _, _, _ = app.example_inputs("cord_test_0014")
    my_image = os.path.join(tempfile.mkdtemp(), "sample_receipt.jpg")
    sample.save(my_image, quality=90)
    print("No upload: using a CORD sample photo.")

ocr_md, ocr_vis, ocr_json, vis, summary, rows, what, my_result = app.run_upload(
    MODEL, my_image, app.OCR_AUTO, None)
display(Markdown(ocr_md))
show(vis, summary, rows, what)

## Step 5: Launch the full interactive UI

This creates a temporary public Gradio URL hosted for the lifetime of this Colab session.
The application code is the same code stored in the public CaliDoc GitHub repository
(`demo/gradio/app.py`). The UI has two tabs: *Try a Real Example* and *Upload Your Own*.

In [ ]:
demo = app.build_demo()
demo.launch(share=True)